# HumanEval Adapter Verification (Colab)

Verify your fine-tuned LoRA adapters by comparing **baseline** (from your existing pipeline CSV) vs **after SFT** (adapter run on HumanEval). No base-model code generation in this notebook.

- **Baseline**: Load `humaneval_pipeline_output.csv` (upload or Google Drive) and compute pass rate from the `status` column.
- **Adapter run**: Load base model + LoRA from `ADAPTER_PATH`, run the same HumanEval evaluation (prompt, generation, tests as in V2_PIPELINE).
- **Paths**: Set via Colab upload: (1) baseline CSV, (2) lora_adapters zip.

## 1. Setup

In [ ]:
# Check GPU (Colab: Runtime -> Change runtime type -> T4/GPU)
!nvidia-smi

In [ ]:
!pip install -q transformers peft datasets torch accelerate tqdm pandas

## 2. Paths

In [ ]:
from google.colab import files
import os
import zipfile

# 1) Upload baseline CSV (humaneval_pipeline_output.csv)
print("Upload: humaneval_pipeline_output.csv")
uploaded_csv = files.upload()
if uploaded_csv:
    BASELINE_CSV_PATH = list(uploaded_csv.keys())[0]
    print(f"Baseline CSV: {BASELINE_CSV_PATH}")
else:
    BASELINE_CSV_PATH = "/content/humaneval_pipeline_output.csv"

# 2) Upload lora_adapters as a zip (contents at top level: adapter_config.json, adapter_model.safetensors, etc.)
print("Upload: lora_adapters zip")
uploaded_adapters = files.upload()
if uploaded_adapters:
    zip_name = list(uploaded_adapters.keys())[0]
    ADAPTER_PATH = "/content/lora_adapters"
    os.makedirs(ADAPTER_PATH, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(ADAPTER_PATH)
    # If zip had a single top-level folder, use it
    subdirs = [d for d in os.listdir(ADAPTER_PATH) if os.path.isdir(os.path.join(ADAPTER_PATH, d))]
    if len(subdirs) == 1 and os.path.isfile(os.path.join(ADAPTER_PATH, subdirs[0], "adapter_config.json")):
        ADAPTER_PATH = os.path.join(ADAPTER_PATH, subdirs[0])
    print(f"Adapters at: {ADAPTER_PATH}")
else:
    ADAPTER_PATH = "/content/lora_adapters"

## 3. Load HumanEval

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("openai/openai_humaneval")
df = pd.DataFrame(ds["test"])
print(f"HumanEval tasks: {len(df)}")
print(df[["task_id", "prompt", "test", "entry_point"]].head(1))

## 4. Load baseline from CSV

In [ ]:
df_baseline = pd.read_csv(BASELINE_CSV_PATH)
passed_baseline = (df_baseline["status"] == "passed").sum()
total_baseline = len(df_baseline)
pass_rate_baseline = passed_baseline / total_baseline if total_baseline else 0
print(f"Baseline (from CSV): {passed_baseline}/{total_baseline} passed, pass@1 = {pass_rate_baseline:.2%}")

## 5. Helper functions

In [ ]:
import re

def construct_prompt_humaneval(docstring_prompt):
    system_message = (
        "You are an expert Python developer. Your task is to complete the function "
        "provided by the user. Follow the docstring exactly. "
        "Provide your output ONLY as a single Python code block starting with ```python."
    )
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": f"Complete this Python function:\n{docstring_prompt}"}
    ]
    return messages

def run_test_humaneval(generated_code, test_script, entry_point):
    execution_code = f"{generated_code}\n\n{test_script}\ncheck({entry_point})"
    try:
        exec_context = {}
        exec(execution_code, exec_context)
        return "passed"
    except AssertionError:
        return "Failed: Logic Hallucination (Assertion Error)"
    except SyntaxError as e:
        return f"Failed: Syntax Hallucination ({e})"
    except Exception as e:
        return f"Failed: {type(e).__name__}: {str(e)}"

def extract_python_code_humaneval(text):
    pattern = r"```(?:python)?\n?(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

## 6. Evaluation function

In [ ]:
import torch
from tqdm import tqdm

def run_humaneval_eval(model, tokenizer, df, device="cuda"):
    model.eval()
    results = []
    pass_count = 0
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="HumanEval"):
        formatted_messages = construct_prompt_humaneval(row["prompt"])
        inputs = tokenizer.apply_chat_template(
            formatted_messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        gen_ids = outputs[0][len(inputs["input_ids"][0]):]
        raw_response = tokenizer.decode(gen_ids, skip_special_tokens=True)
        clean_code = extract_python_code_humaneval(raw_response)
        result = run_test_humaneval(clean_code, row["test"], row["entry_point"])
        passed = result == "passed"
        if passed:
            pass_count += 1
        results.append({"task_id": row["task_id"], "passed": passed, "result": result})
    total = len(df)
    pass_rate = pass_count / total if total else 0
    return results, pass_count, total, pass_rate

## 7. Run with adapters only

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_id = "Qwen/Qwen2.5-Coder-3B-Instruct"
# Tokenizer from adapter path if it has tokenizer files, else from base
try:
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(base_id, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

In [ ]:
results_sft, pass_count_sft, total_sft, pass_rate_sft = run_humaneval_eval(model, tokenizer, df)
print(f"After SFT (adapters): {pass_count_sft}/{total_sft} passed, pass@1 = {pass_rate_sft:.2%}")

## 8. Compare and optional breakdown

In [ ]:
print("=== Pass rate comparison ===")
print(f"Before SFT (from CSV): {pass_rate_baseline:.2%} ({passed_baseline}/{total_baseline})")
print(f"After SFT (adapters):  {pass_rate_sft:.2%} ({pass_count_sft}/{total_sft})")
diff = pass_rate_sft - pass_rate_baseline
print(f"Difference: {diff:+.2%}")
if pass_rate_sft > pass_rate_baseline:
    print("Conclusion: Adapter improves HumanEval pass@1.")
elif pass_rate_sft < pass_rate_baseline:
    print("Conclusion: Adapter pass rate is lower than baseline.")
else:
    print("Conclusion: Same pass rate.")

In [ ]:
# Optional: failure-mode breakdown for SFT run
from collections import Counter
sft_failures = [r["result"] for r in results_sft if not r["passed"]]
failure_counts = Counter(sft_failures)
print("After SFT failure breakdown:")
for msg, count in failure_counts.most_common():
    print(f"  {count:3d}: {msg[:60]}..." if len(msg) > 60 else f"  {count:3d}: {msg}")

## 9. Optional: save adapter run results

In [ ]:
# Save adapter run results to CSV (task_id, passed, result, generated_code)
out_df = pd.DataFrame(results_sft)
out_path = "/content/humaneval_adapter_results.csv"
out_df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

# Download the CSV to your machine (Colab)
from google.colab import files
files.download(out_path)
print("Download started: humaneval_adapter_results.csv")